In [12]:
# ============================================================
# CELL 1 — Setup, Load & Data Validation
# Project: Bangla Cyberbullying Detection & Social Engagement Prediction
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, re, glob, os
pd.set_option('display.max_colwidth', 80)

PROJECT_DIR = '/content/drive/MyDrive/cyberbullying-detection'
RAW_DIR     = f'{PROJECT_DIR}/data/raw'

csv_files = glob.glob(f'{RAW_DIR}/*.csv')
assert csv_files, f'No CSV found in {RAW_DIR}'
CSV_PATH = csv_files[0]
print(f'Loading: {os.path.basename(CSV_PATH)}\n')

df = pd.read_csv(CSV_PATH)

# ---------- 1. Structure ----------
print('='*60); print('1. STRUCTURE')
print(f'shape: {df.shape}')
print(df.dtypes.to_string())

# ---------- 2. Missing values ----------
print('\n'+'='*60); print('2. MISSING VALUES')
print(df.isna().sum().to_string())

# ---------- 3. Corrupted rows ----------
print('\n'+'='*60); print('3. CORRUPTED ROWS')
corrupt = df['Text'].astype(str).str.strip().isin(['#NAME?', '#REF!', '#VALUE!', ''])
print(f'Excel-error / empty Text: {corrupt.sum()}')

# ---------- 4. Duplicates ----------
print('\n'+'='*60); print('4. DUPLICATE ANALYSIS')
txt = df['Text'].astype(str).str.strip()
print(f'fully identical rows      : {df.duplicated().sum()}')
print(f'duplicated Text values    : {txt.duplicated().sum()}')

nlab = df.assign(_t=txt).groupby('_t')['Label'].nunique()
consistent_dup = df.assign(_t=txt)[df.assign(_t=txt)['_t'].isin(nlab[nlab == 1].index) & txt.duplicated(keep=False)]
conflicting    = nlab[nlab > 1].index
print(f'dup Text, SAME label      : {len(consistent_dup)} rows')
print(f'dup Text, CONFLICTING label: {len(conflicting)} unique texts '
      f'({df.assign(_t=txt)["_t"].isin(conflicting).sum()} rows)')
print('\n--- conflicting examples (strategy must be documented, do NOT drop silently) ---')
for t in list(conflicting)[:5]:
    print(f'  {t[:55]!r} -> {df.assign(_t=txt).query("_t == @t")["Label"].tolist()}')

# ---------- 5. Text length ----------
print('\n'+'='*60); print('5. TEXT LENGTH')
wc = txt.str.split().str.len()
print(f'chars  -> {df["Text"].astype(str).str.len().describe()[["mean","50%","max"]].round(1).to_dict()}')
print(f'words  -> {wc.describe()[["mean","50%","max"]].round(1).to_dict()}')
print(f'<=2 words: {(wc <= 2).sum()}   <=1 word: {(wc <= 1).sum()}')

# ---------- 6. Class balance ----------
print('\n'+'='*60); print('6. CLASS BALANCE (Objective 1 target)')
vc = df['Label'].value_counts()
print(pd.DataFrame({'count': vc, 'pct': (vc/len(df)*100).round(2)}).to_string())
print(f'imbalance ratio (max/min): {vc.max()/vc.min():.1f} : 1')

# ---------- 7. Reaction count ----------
print('\n'+'='*60); print('7. REACTION COUNT (Objective 2 target)')
y = df['Comment React Number']
print(f'zeros : {(y==0).sum()} ({(y==0).mean()*100:.1f}%)   <=1 : {(y<=1).mean()*100:.1f}%   <=3 : {(y<=3).mean()*100:.1f}%')
print(f'unique values: {y.nunique()}   max: {y.max():.0f}')
print(y.quantile([.5,.75,.9,.95,.99,1.0]).to_string())

# ---------- 8. Metadata (EDA only — NOT model features) ----------
print('\n'+'='*60); print('8. METADATA (EDA only)')
print(f"Category: {df['Category'].value_counts().to_dict()}")
print(f"Gender  : {df['Gender'].value_counts().to_dict()}   <- check casing inconsistency")

# ---------- 9. Noise indicators ----------
print('\n'+'='*60); print('9. NOISE INDICATORS')
print(f"leading/trailing whitespace : {(df['Text'].astype(str) != txt).sum()}")
print(f"contains URL                : {txt.str.contains(r'http|www\\.').sum()}")
print(f"contains newline            : {txt.str.contains('\\n').sum()}")
print(f"rows with <50% Bangla chars : "
      f"{txt.map(lambda s: len(re.findall(r'[\u0980-\u09FF]', s)) / max(len(re.sub(r'\\s','',s)), 1) < 0.5).sum()}")

print('\n'+'='*60)
print('Validation complete. df loaded UNMODIFIED — cleaning in next cell.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading: Cyberbulling Bangla Dataset.csv

1. STRUCTURE
shape: (44001, 5)
Text                     object
Category                 object
Gender                   object
Comment React Number    float64
Label                    object

2. MISSING VALUES
Text                    0
Category                0
Gender                  0
Comment React Number    3
Label                   0

3. CORRUPTED ROWS
Excel-error / empty Text: 7

4. DUPLICATE ANALYSIS
fully identical rows      : 142
duplicated Text values    : 601
dup Text, SAME label      : 1008 rows
dup Text, CONFLICTING label: 48 unique texts (109 rows)

--- conflicting examples (strategy must be documented, do NOT drop silently) ---
  '#NAME?' -> ['Harassment', 'Harassment', 'Harassment', 'Harassment', 'Hate Speech', 'Neutral', 'Harassment']
  'অবশেষে মুক্তি পেল বিলাশবাড়ীর আবু রায়হান মামুর অভিনীত শর' -> ['Neu

In [13]:
# ============================================================
# CELL 2 — Cleaning & Documented Duplicate/Conflict Resolution
# Input : df (raw, unmodified from Cell 1)
# Output: df_clean + audit trail + archived conflict set
# ============================================================
import unicodedata, os

PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

audit = {'raw_rows': len(df)}
d = df.copy()

# ---- STEP 1: corrupted rows (Excel formula errors) ----
corrupt_mask = d['Text'].astype(str).str.strip().isin(['#NAME?', '#REF!', '#VALUE!', '', 'nan'])
audit['dropped_corrupted'] = int(corrupt_mask.sum())
d = d[~corrupt_mask].copy()

# ---- STEP 2: Unicode NFC + whitespace normalization ----
# Bangla conjuncts exist in composed/decomposed forms; without NFC
# identical-looking texts will not match during deduplication.
def normalize(s):
    s = unicodedata.normalize('NFC', str(s))
    s = s.replace('​', '').replace('‌', '').replace('‍', '')  # zero-width chars
    return re.sub(r'\s+', ' ', s).strip()

d['Text'] = d['Text'].map(normalize)
audit['rows_after_normalize'] = len(d)

# ---- STEP 3: metadata casing (EDA only, never a model feature) ----
d['Gender']   = d['Gender'].str.strip().str.capitalize()
d['Category'] = d['Category'].str.strip().str.capitalize()

# ---- STEP 4: missing reaction counts ----
audit['react_missing'] = int(d['Comment React Number'].isna().sum())
d['Comment React Number'] = d['Comment React Number'].fillna(0).astype(int)

# ---- STEP 5: CONFLICTING-LABEL RESOLUTION (documented, not silent) ----
nlab = d.groupby('Text')['Label'].nunique()
conflict_texts = set(nlab[nlab > 1].index)
audit['conflicting_texts'] = len(conflict_texts)
audit['conflicting_rows']  = int(d['Text'].isin(conflict_texts).sum())

# archive the FULL conflicting set before resolving anything
conflict_df = d[d['Text'].isin(conflict_texts)].sort_values('Text')
conflict_df.to_csv(f'{PROCESSED_DIR}/conflicting_labels.csv', index=False)

# strategy: majority vote; drop only on a tie (no annotation consensus)
resolved, tie_texts = {}, []
for t, grp in d[d['Text'].isin(conflict_texts)].groupby('Text'):
    vc = grp['Label'].value_counts()
    if len(vc) > 1 and vc.iloc[0] == vc.iloc[1]:
        tie_texts.append(t)
    else:
        resolved[t] = vc.index[0]

d['Label'] = d.apply(lambda r: resolved.get(r['Text'], r['Label']), axis=1)
rows_before_tie_drop = len(d)
d = d[~d['Text'].isin(tie_texts)].copy()

audit['conflicts_majority_resolved'] = len(resolved)
audit['conflicts_dropped_tie']       = len(tie_texts)
audit['rows_dropped_tie']            = rows_before_tie_drop - len(d)

# ---- STEP 6: duplicate Text removal (AFTER normalization, BEFORE split) ----
before = len(d)
d = d.drop_duplicates(subset='Text', keep='first').reset_index(drop=True)
audit['dropped_duplicate_text'] = before - len(d)

# ---- STEP 7: empty / very short ----
audit['empty_removed'] = int((d['Text'].str.len() == 0).sum())
d = d[d['Text'].str.len() > 0].reset_index(drop=True)
wc = d['Text'].str.split().str.len()
audit['kept_1word'] = int((wc <= 1).sum())   # KEPT: short slurs carry strong signal
audit['kept_2word'] = int((wc <= 2).sum())

# ---- STEP 8: save + report ----
df_clean = d
df_clean.to_csv(f'{PROCESSED_DIR}/cleaned.csv', index=False)
audit['final_rows']    = len(df_clean)
audit['total_removed'] = audit['raw_rows'] - audit['final_rows']

print('=' * 60); print('CLEANING AUDIT TRAIL')
for k, v in audit.items():
    print(f'  {k:32s}: {v}')

print('\n' + '=' * 60); print('TIE CASES DROPPED (no annotation consensus)')
for t in tie_texts[:10]:
    print(f'  {t[:60]!r}')
print(f'  ... total {len(tie_texts)}')

print('\n' + '=' * 60); print('CLASS BALANCE AFTER CLEANING')
vc = df_clean['Label'].value_counts()
print(pd.DataFrame({'count': vc, 'pct': (vc / len(df_clean) * 100).round(2)}).to_string())

print(f'\nSaved -> {PROCESSED_DIR}/cleaned.csv')
print(f'Saved -> {PROCESSED_DIR}/conflicting_labels.csv  (manual review / thesis appendix)')

CLEANING AUDIT TRAIL
  raw_rows                        : 44001
  dropped_corrupted               : 7
  rows_after_normalize            : 43994
  react_missing                   : 3
  conflicting_texts               : 52
  conflicting_rows                : 113
  conflicts_majority_resolved     : 5
  conflicts_dropped_tie           : 47
  rows_dropped_tie                : 94
  dropped_duplicate_text          : 755
  empty_removed                   : 0
  kept_1word                      : 1049
  kept_2word                      : 3785
  final_rows                      : 43145
  total_removed                   : 856

TIE CASES DROPPED (no annotation consensus)
  'অবশেষে মুক্তি পেল বিলাশবাড়ীর আবু রায়হান মামুর অভিনীত শর্ট '
  'অভিনেত্রী মানেই"সেচ্ছায় ধর্ষিতা" এবং "মর্ডান বেশ্যা" ! তারা'
  'অভিনেত্রী মানেইসেচ্ছায় "ধর্ষিতা" এবং মর্ডান "বেশ্যা" ! তারা'
  'অরে বাটপার'
  'অসভ্য মেয়ে'
  'আচোদা'
  'আমার ক্রাশ'
  'এর একাধিক যৌন সংগি আছে এ শয়তান মেয়ে তার বাবার সাথেও অপকর্ম'
  'ওরে বাড়া'
  'খাওয